# Experiment 2 -- aggregate and plot: heatmaps over (Delta_T, Delta_S) (new pooled method)

Identical to `experiment2_00_aggregate_and_plot.ipynb`, except the "pooled"
column uses `method="new_pooled"` (`target_source_pooled_subspace_estimate`)
instead of `method="pooled"` (`pooled_subspace_estimate`, the original
data-pooling estimator) -- via a single filter/rename right after loading
the raw data. Every other cell is untouched and keys off the literal string
`"pooled"` regardless of which estimator actually produced it.

**`new_pooled` was wired into `run_experiment2.py` but the Slurm array has
not been rerun** -- until it is, this notebook's `D_pooled` column will be
empty (the sanity-check cell below will show 0 seeds for `pooled` here).

Reads every per-task CSV written by `run_experiment2.py` into
`Results_simulation/experiment2/raw/`, aggregates the Monte Carlo mean error
at each `(regime, Delta_T, Delta_S, method)` grid point, and produces the
3x4 heatmap figure: rows = regimes R1-R3, columns = {target-only raw error,
source-only minus target-only, pooled minus target-only, adaptive minus
target-only}.

Run this after the SLURM array in `Slurm_Scripts/experiment2_heatmap/`
has finished (or partially finished).

In [19]:
import sys, os, glob

# Hardcoded (rather than relative to "..") because the kernel's cwd isn't
# guaranteed to be this notebook's directory -- e.g. VS Code's Jupyter
# extension often starts kernels from the workspace root instead. Mirrors
# the PROJECT_ROOT convention already used in Slurm_Scripts/*/run_*.sh and
# experiment1_01_aggregate_and_plot.ipynb.
PROJECT_ROOT = "/home/nandy.15/Research/Transfer_clustering"
sys.path.insert(0, os.path.join(PROJECT_ROOT, "Experiments_Script"))

# The figure below uses matplotlib's text.usetex=True, which shells out to
# `latex`/`dvipng`. Hardcoded (rather than relying on `module load texlive`
# having been run in the launching shell) because the Jupyter kernel's PATH
# is whatever VS Code/Jupyter started with -- it won't see a module loaded
# after the fact. Matches the cluster's `module show texlive/2025`.
TEXLIVE_BIN = "/opt/lmod/texlive/2025/bin/x86_64-linux"
if TEXLIVE_BIN not in os.environ["PATH"].split(os.pathsep):
    os.environ["PATH"] = TEXLIVE_BIN + os.pathsep + os.environ["PATH"]

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from run_experiment2 import REGIME_ORDER, DELTA_T_GRID, DELTA_S_GRID, REGIMES

In [ ]:
RAW_DIR = os.path.join(PROJECT_ROOT, "Results_simulation", "experiment2", "raw")
COMBINED_DIR = os.path.join(PROJECT_ROOT, "Results_simulation", "experiment2", "combined_new_pooled")
os.makedirs(COMBINED_DIR, exist_ok=True)

paths = sorted(glob.glob(os.path.join(RAW_DIR, "*.csv")))
print(f"Found {len(paths)} raw result files")
assert paths, f"No CSVs found in {RAW_DIR} -- has the SLURM array finished any tasks yet?"

df = pd.concat([pd.read_csv(p) for p in paths], ignore_index=True)
df.head()

In [ ]:
# Swap in the new pooled method: drop the original "pooled" rows, then
# rename "new_pooled" -> "pooled" so every downstream cell (which keys off
# the literal string "pooled" -- pivot_table columns, D_pooled, plotting)
# is unchanged and now reflects target_source_pooled_subspace_estimate.
df = df[df["method"] != "pooled"].copy()
df.loc[df["method"] == "new_pooled", "method"] = "pooled"
df.head()

In [21]:
# Sanity check: how many distinct seeds actually landed per (regime, method)?
# If well short of N_SEEDS in run_experiment2.py, the array job hasn't
# finished (or some tasks failed -- check Slurm_Scripts/.../Error_Messages).
df.groupby(["regime", "method"])["seed"].nunique().unstack()

method,adaptive,pooled,source,target
regime,,,,
R1,100,100,100,100
R2,100,100,100,100
R3,100,100,100,100


In [22]:
# Err_M(Delta_T, Delta_S) (Sec 4.6): Monte Carlo mean error per grid point.
err = (
    df.groupby(["regime", "Delta_T", "Delta_S", "method"])["error"]
      .agg(mean_error="mean", se_error=lambda s: s.std(ddof=1) / np.sqrt(len(s)), n="count")
      .reset_index()
)
err.to_csv(os.path.join(COMBINED_DIR, "experiment2_err_by_method.csv"), index=False)

# D_M(Delta_T, Delta_S) := Err_M - Err_target for M in {source, pooled, adaptive} (Sec 4.6).
wide = err.pivot_table(index=["regime", "Delta_T", "Delta_S"], columns="method", values="mean_error").reset_index()
wide["D_source"] = wide["source"] - wide["target"]
wide["D_pooled"] = wide["pooled"] - wide["target"]
wide["D_adaptive"] = wide["adaptive"] - wide["target"]
wide.to_csv(os.path.join(COMBINED_DIR, "experiment2_summary.csv"), index=False)
wide.head(12)

method,regime,Delta_T,Delta_S,adaptive,pooled,source,target,D_source,D_pooled,D_adaptive
0,R1,0.5,0.5,0.47335,0.47295,0.47325,0.46965,0.00360,0.00330,0.00370
1,R1,0.5,1.0,0.47015,0.47460,0.46680,0.46965,-0.00285,0.00495,0.00050
2,R1,0.5,1.5,0.46990,0.46890,0.46705,0.46965,-0.00260,-0.00075,0.00025
3,R1,0.5,2.0,0.45590,0.46375,0.44665,0.46965,-0.02300,-0.00590,-0.01375
4,R1,0.5,2.5,0.43145,0.43220,0.41055,0.46965,-0.05910,-0.03745,-0.03820
5,R1,1.0,0.5,0.46830,0.46635,0.47225,0.46070,0.01155,0.00565,0.00760
6,R1,1.0,1.0,0.46700,0.46450,0.47165,0.46070,0.01095,0.00380,0.00630
7,R1,1.0,1.5,0.46195,0.45150,0.46420,0.46070,0.00350,-0.00920,0.00125
8,R1,1.0,2.0,0.41740,0.42455,0.40100,0.46070,-0.05970,-0.03615,-0.04330
9,R1,1.0,2.5,0.36855,0.34550,0.33060,0.46070,-0.13010,-0.11520,-0.09215


## Figure: 3x4 heatmap grid

Rows = regimes R1-R3. Columns = target-only raw error (sequential blue
colorscale), source-only minus target-only, pooled minus
target-only, adaptive minus target-only (the latter three diverging, blue =
improvement / negative, red = worse / positive, centered at zero, per Sec
4.8 conventions (1)-(4)). Columns 2-4 share one common color scale across
all three regimes (convention (5)); the shared range is the 98th
percentile of |D| across every regime so a handful of extreme cells don't
wash out the rest of the scale (convention (6)) -- values beyond that are
clipped and shown via the colorbar's triangular extension arrows.

In [ ]:
def to_grid(sub, value_col):
    """(regime-filtered) long dataframe -> len(DELTA_S_GRID) x len(DELTA_T_GRID) array,
    rows = Delta_S (ascending), cols = Delta_T (ascending), matching the
    plan's axis convention (horizontal = Delta_T, vertical = Delta_S)."""
    pivot = sub.pivot(index="Delta_S", columns="Delta_T", values=value_col)
    pivot = pivot.reindex(index=DELTA_S_GRID, columns=DELTA_T_GRID)
    return pivot.values

# Shared color scale for columns 2-4 (convention 5), robust to outliers (convention 6).
all_D = np.concatenate([wide["D_source"].values, wide["D_pooled"].values, wide["D_adaptive"].values])
D_LIM = float(np.nanpercentile(np.abs(all_D), 98))

# Shared sequential scale for column 1 across regimes.
ERR_MAX = float(wide["target"].max())

plt.rcParams.update({
    "text.usetex": True,
    "font.family": "serif",
    "font.size": 10,
    "axes.labelsize": 10,
    "axes.titlesize": 10,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
    "legend.fontsize": 10,
})

fig, axes = plt.subplots(3, 4, figsize=(17, 11))

col_specs = [
    ("target", r"$\widehat{\mathcal{L}}_{T}$", "Blues", 0.0, ERR_MAX, False),
    ("D_source", r"$\widehat{\mathcal{E}}_S$", "RdBu_r", -D_LIM, D_LIM, True),
    ("D_pooled", r"$\widehat{\mathcal{E}}_{\mathrm{pool}}$", "RdBu_r", -D_LIM, D_LIM, True),
    ("D_adaptive", r"$\widehat{\mathcal{E}}_{\mathrm{adp}}$", "RdBu_r", -D_LIM, D_LIM, True),
]

for row, regime in enumerate(REGIME_ORDER):
    sub = wide[wide["regime"] == regime]
    cfg = REGIMES[regime]
    for col, (value_col, title, cmap, vmin, vmax, diverging) in enumerate(col_specs):
        ax = axes[row, col]
        grid = to_grid(sub, value_col)
        im = ax.imshow(grid, origin="lower", aspect="auto", cmap=cmap, vmin=vmin, vmax=vmax)
        ax.set_xticks(range(len(DELTA_T_GRID)))
        ax.set_xticklabels(DELTA_T_GRID, rotation=45)
        ax.set_yticks(range(len(DELTA_S_GRID)))
        ax.set_yticklabels(DELTA_S_GRID)
        if row == 0:
            ax.set_title(title)
        if col == 0:
            ax.set_ylabel(rf"\texttt{{{regime}}} ($d={cfg['d']}$, $n_T={cfg['n_T']}$, $n_S={cfg['n_S']}$)" + "\n" + r"$\Delta_S$")
        if row == 2:
            ax.set_xlabel(r"$\Delta_T$")
        # One shared colorbar per column (attached to the bottom row).
        if row == 2:
            cbar = fig.colorbar(im, ax=axes[:, col].tolist(), orientation="horizontal",
                                 fraction=0.04, pad=0.08, extend=("both" if diverging else "neither"))
            cbar.ax.tick_params(labelsize=10)

fig.suptitle(r"Experiment 2: error over $(\Delta_T, \Delta_S)$, $\mu = 0.8$ fixed (new pooled method)", y=0.93)
fig.savefig(os.path.join(COMBINED_DIR, "experiment2_heatmaps_new_pooled.pdf"), bbox_inches="tight")
plt.show()

## Diagnostic: calibrated C0 stability

As in Experiment 1, `run_experiment2.py` records `C0_used` for every
`adaptive` row (the bootstrap-calibrated constant from
`calibrate_C0_bootstrap`). Since Experiment 2's grid covers a much wider
range of `(Delta_T, Delta_S)` than Experiment 1, this is a useful check on
whether `C0_hat` stays stable across signal strengths too, not just
across `(n_T, d)`.

In [24]:
c0 = df[df["method"] == "adaptive"].copy()
c0["C0_used"] = pd.to_numeric(c0["C0_used"], errors="coerce")
c0.groupby("regime")["C0_used"].agg(["mean", "std", "count"])

,mean,std,count
regime,,,
R1,1.210904,0.018575,2500
R2,1.210904,0.018575,2500
R3,1.029965,0.003069,2500
